# Preprocessing: BPE tokenization

Dataset: `cardiffnlp/tweet_eval` (emotion), about 3.3k training tweets.

Preprocessing is kept minimal: punctuation is kept, and there is no stopword removal, stemming or lemmatisation (tweets are short, noisy and full of misspellings). We compare BPE tokenizers over a grid of hyperparameters:
- **`lowercase`** or keep the original case (capitals can signal intensity);
- **`vocab_size`**: number of tokens in the vocabulary;
- **`min_frequency`**: a pair of tokens is only merged if it appears at least this many times;
- **`pre-tokenizer`**: `Metaspace` (tutorial default) splits only on blanks and marks each word start with `▁`, so `sad!!` becomes the chunk `▁sad!!`. `Whitespace` also splits punctuation (`sad`, `!!`) but drops the blanks.

The tokenizer is trained with `part1.experiment.train_tokenizer` from the tutorial.

In [1]:
import sys
from pathlib import Path

# Assuming the working directory is root/part1
sys.path.insert(0, str(Path.cwd().parent))

import part1
from datasets import load_dataset

dataset = load_dataset("cardiffnlp/tweet_eval", "emotion")
emotion_outcome_space = part1.clslib.NamedClasses(dataset["train"].features["label"].names)
example_tweet = dataset["train"][0]["text"]

print(dataset)
print(emotion_outcome_space)
print("Original:", example_tweet)

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cardiffnlp/tweet_eval/resolve/main/README.md
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cardiffnlp/tweet_eval/resolve/main/README.md
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cardiffnlp/tweet_eval/resolve/main/README.md
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cardiffnlp/tweet_eval/resolve/main/README.md
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cardiffnlp/tweet_eval/resolve/main/README.md
Retrying in 8s [Retry 5/5].


README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

c:\Users\damla\OneDrive\Desktop\uni 25-26\machine learning and data mining 2\project\nlp_first_assignment\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\damla\.cache\huggingface\hub\datasets--cardiffnlp--tweet_eval. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\damla\OneDrive\Desktop\u

emotion/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  233kB            

emotion/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

emotion/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  105kB            

... (3 outputs left out)

Generating train split:   0%|          | 0/3257 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1421 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/374 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 3257
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1421
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 374
    })
})
0: anger, 1: joy, 2: optimism, 3: sadness
Original: “Worry is a down payment on a problem you may never have'.  Joyce Meyer.  #motivation #leadership #worry


## BPE tokenizer

The tokenizer is trained only on the training split.

In [2]:
from tokenizers.pre_tokenizers import Metaspace, Whitespace

PRETOKENIZERS = {"metaspace": Metaspace, "whitespace": Whitespace}


class BPEPreprocessor:
    """Optional lowercasing + BPE tokenization.

    Call `fit(train_texts)` first, then `tokenize(text)` returns the BPE tokens.
    """

    def __init__(self, lowercase=True, vocab_size=5000, min_frequency=2, pretokenizer="metaspace"):
        self.lowercase = lowercase
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency
        self.pretokenizer = pretokenizer
        self.tokenizer = None

    def fit(self, train_texts):
        corpus = [t.lower() for t in train_texts] if self.lowercase else list(train_texts)
        self.tokenizer = part1.experiment.train_tokenizer(
            corpus, self.vocab_size, self.min_frequency, pretokenizer=PRETOKENIZERS[self.pretokenizer]()
        )
        return self

    def tokenize(self, text):
        return self.tokenizer.encode(text.lower() if self.lowercase else text).tokens

    def __len__(self):
        """Number of tokens in the learned vocabulary."""
        return self.tokenizer.get_vocab_size()

## Hyperparameter grid

Every combination below is trained on the training split only.

In [3]:
import itertools
from tqdm.auto import tqdm

GRID = {
    "lowercase": [True, False],
    "vocab_size": [500, 1000, 2000],
    "min_frequency": [2, 5],
    "pretokenizer": ["metaspace", "whitespace"],
}
configs = [dict(zip(GRID, values)) for values in itertools.product(*GRID.values())]
print(len(configs), "configurations")

train_texts = dataset["train"]["text"]
preprocessors = [BPEPreprocessor(**cfg).fit(train_texts) for cfg in tqdm(configs)]

print("Original:", example_tweet)
for cfg, prep in zip(configs, preprocessors):
    print(cfg, "->", prep.tokenize(example_tweet))


def find(**wanted):
    """Return the fitted preprocessor matching the given hyperparameters."""
    return next(p for c, p in zip(configs, preprocessors) if all(c[k] == v for k, v in wanted.items()))

24 configurations


  0%|          | 0/24 [00:00<?, ?it/s]

Original: “Worry is a down payment on a problem you may never have'.  Joyce Meyer.  #motivation #leadership #worry
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 2, 'pretokenizer': 'metaspace'} -> ['▁', '“', 'w', 'or', 'ry', '▁is', '▁a', '▁d', 'ow', 'n', '▁p', 'ay', 'm', 'ent', '▁on', '▁a', '▁p', 'ro', 'ble', 'm', '▁you', '▁m', 'ay', '▁ne', 'ver', '▁have', "'", '.', '▁', '\xa0', 'j', 'o', 'y', 'ce', '▁me', 'y', 'er', '.', '▁', '▁#m', 'ot', 'i', 'v', 'at', 'ion', '▁#', 'le', 'ad', 'ers', 'h', 'i', 'p', '▁#', 'w', 'or', 'ry']
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 2, 'pretokenizer': 'whitespace'} -> ['“', 'wor', 'ry', 'is', 'a', 'do', 'wn', 'p', 'ay', 'm', 'ent', 'on', 'a', 'pro', 'ble', 'm', 'you', 'm', 'ay', 'ne', 'ver', 'have', "'", '.', 'jo', 'y', 'ce', 'me', 'y', 'er', '.', '#', 'mo', 'ti', 'v', 'ation', '#', 'le', 'ad', 'er', 'sh', 'i', 'p', '#', 'wor', 'ry']
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 5, 'pretokenizer': 'metaspace'} -> ['▁', '

## Inspecting every vocabulary

- **longest tokens**: whole words, hashtags or emojis it managed to keep in one piece;
- **first learned tokens**: ids right after the initial alphabet are the earliest (most frequent) merges;
- **tokens with symbols**: how punctuation and hashtags are handled.

In [4]:
import statistics

clean = lambda t: t.lstrip("\u2581")  # drop the Metaspace marker when measuring length

for cfg, prep in zip(configs, preprocessors):
    vocab = prep.tokenizer.get_vocab()  # token -> id
    by_id = sorted(vocab, key=vocab.get)

    print("=" * 100)
    print(cfg)
    lengths = [len(clean(t)) for t in vocab]
    print(f"token length: mean={statistics.mean(lengths):.2f}  median={statistics.median(lengths)}  max={max(lengths)}")
    print("longest tokens:", sorted(vocab, key=lambda t: -len(clean(t)))[:25])
    print("tokens with symbols:", [t for t in by_id if len(clean(t)) > 1 and not clean(t).replace("'", "").isalnum()][:30])

{'lowercase': True, 'vocab_size': 500, 'min_frequency': 2, 'pretokenizer': 'metaspace'}
token length: mean=1.54  median=1.0  max=5
longest tokens: ['[UNK]', '▁@user', "▁don't", '▁about', '▁some', "▁it's", 'king', '▁have', '▁with', '▁that', 'user', '▁what', '▁this', '▁just', 'ight', '▁your', '▁they', '▁when', 'ould', '▁like', 'ress', 'ally', 'all', 'ack', 'ion']
tokens with symbols: ['[UNK]', '▁@user', '..', '▁#s', '\\n', '!!', '...', '▁#b', '▁#h', '▁#f', '▁#m']
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 2, 'pretokenizer': 'whitespace'}
token length: mean=1.68  median=1.0  max=6
longest tokens: ['people', 'about', 'press', '[UNK]', 'thing', 'ation', 'what', 'just', 'that', 'ight', 'ould', 'user', 'ever', 'ting', 'they', 'king', 'pres', 'tion', 'peop', 'from', 'like', 'than', 'your', 'will', 'when']
tokens with symbols: ['[UNK]', '..', '!!', '...']
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 5, 'pretokenizer': 'metaspace'}
token length: mean=1.54  median=1.0  max

## Default tokenizer

Shared configuration to start from: lowercase, `Whitespace` pre-tokenizer, 2000 tokens, `min_frequency=2`. Use `default_prep` to tokenize new text, and `preprocessed` (train/validation/test) for the already tokenized dataset with its labels.

In [5]:
default_prep = BPEPreprocessor(lowercase=True, vocab_size=2000, min_frequency=2, pretokenizer="whitespace").fit(train_texts)

# Whole dataset, preprocessed: tokens and labels per split (same fitted tokenizer for all splits)
preprocessed = {
    split: {
        "tokens": [default_prep.tokenize(t) for t in dataset[split]["text"]],
        "labels": dataset[split]["label"],
    }
    for split in dataset
}
print({split: len(d["tokens"]) for split, d in preprocessed.items()})
print(preprocessed["train"]["tokens"][0], preprocessed["train"]["labels"][0])

{'train': 3257, 'test': 1421, 'validation': 374}
['“', 'worry', 'is', 'a', 'down', 'pay', 'ment', 'on', 'a', 'problem', 'you', 'may', 'never', 'have', "'", '.', 'joy', 'ce', 'me', 'y', 'er', '.', '#', 'motiv', 'ation', '#', 'lead', 'er', 'ship', '#', 'worry'] 2


# Model Architectures

This notebook contains the implementations of the three text classification architectures used in our first project:

1. Log-linear softmax classifier

2. Single-hidden-layer Multilayer Perceptron (MLP)

3. -


In [18]:
import scipy
from scipy.special import softmax
import numpy as np
from tokenizers import Tokenizer

In [12]:
class Encoder:
    """Base class for text feature encoders."""

    def __len__(self):
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError


class ProbabilisticModel:
    """Base class for probabilistic classifiers."""

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

**1. Log-Linear Softmax Classifier**

#explanation

In [13]:
class LogLinearSoftmaxRegressor(ProbabilisticModel):

    def __init__(self, encoder, W: 'np.array', b: 'np.array'):
        """
        encoder: turns a text (str) into a D-dimensional vector
        W: KxD matrix of weights
        b: B-dimensional vector of weights
        """
        self.encoder = encoder
        assert W.shape[0] == len(b), f"The matrix W should have shape KxD with K={len(b)}"
        assert W.shape[1] == len(encoder), f"The matrix W should have shape KxD with D={len(encoder)}"
        self.W = W
        self.b = b
        
    def __call__(self, x: str) -> 'np.array':
        h = self.encoder(x) # D features
        s = np.dot(self.W, h) + self.b # K linear predictors
        pmf = softmax(s) # K probabilities
        return pmf

**2. Single-Hidden-Layer MLP**

#explanation

In [14]:
#single layer mlp: Teo

3.

In [15]:
#another implementation: Samuel

# Representation Spaces

1. Token Indicators
2. Bag of Words
3. TF-IDF

1. Token Indicators

#explanation


In [16]:
#code

2. Bag of Words(BoW)

Bag-of-Words is a method for converting text into numerical feature vectors. It represents each text by counting how often each token appears, without considering the order in which the tokens occur. 

In our project, we use a trained Byte-Pair Encoding (BPE) tokenizer to split tweets into tokens, which may include complete words, subwords, and punctuation. Each tweet is represented by a fixed-length vector, where each position corresponds to a token in the vocabulary and its value indicates how frequently that token occurs in the tweet.

In [24]:
class BagOfWords(Encoder):

    def __init__(self, preprocessor: BPEPreprocessor):
        self.prep = preprocessor
        self.tokenizer = preprocessor.tokenizer
        self._dim = self.tokenizer.get_vocab_size()

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return self._dim

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a vector counting each token's occurrences"""
        text = x.lower() if self.prep.lowercase else x
        token_ids = self.tokenizer.encode(text).ids
        vector = np.zeros(len(self))
        np.add.at(vector, token_ids, 1)
        return vector

3. TF-IDF 

#explanation

In [ ]:
#code

# Experiments